# Preprocesamiento Audi A3 (Alemania)

**Rol de canalización:** Cuaderno obligatorio 02, el paso de preparación de datos sin procesar a procesados.

Este cuaderno limpia el CSV sin procesar y guarda un conjunto de datos procesados ​​con un esquema fijo. Las transformaciones son intencionalmente explícitas para que los estudiantes puedan ver cómo los campos extraídos sin procesar se convierten en columnas listas para SQL.

**Consume:** el último CSV sin procesar de `data/raw`, producido por Notebook 01.
**Produce:** un CSV procesado con marca de tiempo en `data/processed` con nombres y tipos de columnas estables.
**Feeds:** BigQuery preparación, objetos de almacén SQL, cuadernos ML y laboratorios opcionales.

Preservamos cada transformación que afecta el conjunto de datos procesados. La EDA sin salida se mantiene fuera de esta ruta principal para que el flujo de referencia siga siendo breve y claro.

### Descripción general funcional
Entradas: el raspado sin procesar más reciente CSV.
Proceso: validar esquema, limpiar tipos, estandarizar texto, marcar valores atípicos y problemas lógicos, y construir un esquema final.
Salidas: un CSV procesado en `data/processed` con nombres y tipos de columnas estables.
Motivo: los pasos posteriores SQL y ML requieren valores numéricos limpios y etiquetas categóricas consistentes.

**Objetivo:** Convertir datos extraídos sin procesar en un conjunto de datos analíticos limpios.
**Entradas:** El último CSV sin procesar de `data/raw`.
**Proceso:** Validar esquema, limpiar tipos, normalizar texto, eliminar valores atípicos/problemas.
**Salidas:** Un CSV procesado en `data/processed` con un esquema estable.
**Por qué:** Se requieren datos limpios y coherentes para las consultas SQL y los modelos ML.


## 1. Importaciones y configuración

Las importaciones son intencionalmente pequeñas para que los principiantes puedan concentrarse en la limpieza de datos en lugar de en la administración de la biblioteca.

### ¿Qué pasa aquí?
Importamos pandas para el trabajo de datos y algunas utilidades para rutas y marcas de tiempo. Mantener las importaciones breves ayuda a los principiantes a centrarse en los pasos de datos en lugar de en las bibliotecas.

### Por qué las importaciones son mínimas
Los cuadernos para principiantes deberían reducir las distracciones. Importamos solo lo que usamos: pandas para tablas, Ruta para rutas de archivos y fecha y hora para nombres de archivos de salida.

**Objetivo:** Cargar solo las bibliotecas necesarias para el preprocesamiento.
**Entradas:** pandas, Ruta, fecha y hora.
**Proceso:** Importar módulos utilizados más adelante en el cuaderno.
**Salidas:** Bibliotecas listas para limpiar y guardar datos.
**Por qué:** Las importaciones mínimas reducen la distracción para los principiantes.


In [ ]:
# Objetivo: Importar las bibliotecas mínimas necesarias para el preprocesamiento y el guardado.  #Objetivo
# Entrada: pandas para manipulación de datos, Ruta para rutas de archivos, fecha y hora para marcas de tiempo.  #Aporte
# Entrada: project_config.yaml compartido para los valores predeterminados de alcance y ruta.  #Aporte
# Proceso: importar bibliotecas, cargar configuraciones y definir directorios relativos al repositorio.  #Proceso
# Salida: Módulos importados y variables de configuración disponibles para celdas posteriores.  #Producción

import pandas as pd  # pandas proporciona operaciones de DataFrame para limpiar y guardar.
from pathlib import Path  # Path proporciona manejo de rutas de archivos independiente del sistema operativo.
from datetime import datetime  # datetime proporciona marcas de tiempo para los nombres de archivos de salida.


def find_repo_root(start: Path) -> Path:  # Localice la raíz del repositorio comprobando las carpetas principales.
    for p in [start] + list(start.parents):
        if (p / '.git').exists() or (p / 'config' / 'project_config.yaml').exists():
            return p
    return start


def load_project_config(config_path: Path) -> dict:  # Lector YAML ligero para clave: configuración de valor.
    config = {}
    if not config_path.exists():
        return config
    for raw_line in config_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or ':' not in line:
            continue
        key, value = line.split(':', 1)
        key = key.strip()
        value = value.strip()
        if value.startswith(('"', "'")) and value.endswith(('"', "'")):
            value = value[1:-1]
        elif value.lower() in ('true', 'false'):
            value = value.lower() == 'true'
        else:
            try:
                value = int(value)
            except ValueError:
                try:
                    value = float(value)
                except ValueError:
                    pass
        config[key] = value
    return config


def resolve_repo_path(repo_root: Path, config_value: str, default_path: str) -> Path:
    path_value = str(config_value or default_path)
    p = Path(path_value)
    return p if p.is_absolute() else (repo_root / p)


repo_root = find_repo_root(Path.cwd())
project_config = load_project_config(repo_root / 'config' / 'project_config.yaml')

make = str(project_config.get('make', 'audi')).strip().lower()
model = str(project_config.get('model', 'a3')).strip().lower()
country_name = str(project_config.get('country', 'germany')).strip().lower()
tag = f'{make}_{model}_{country_name}'

RAW_DATA_DIR = resolve_repo_path(repo_root, project_config.get('raw_data_path', 'data/raw'), 'data/raw')
PROCESSED_DATA_DIR = resolve_repo_path(repo_root, project_config.get('processed_data_path', 'data/processed'), 'data/processed')
REPORTS_DIR = repo_root / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


## 2. Localice y cargue el último archivo sin formato

La canalización siempre carga el último raspado por marca de tiempo de nombre de archivo para mantener la coherencia con el flujo del curso.

### Cómo se selecciona el archivo sin formato
Encontramos la raíz del proyecto, luego cargamos el CSV sin formato más reciente que coincida con el patrón de nombre de archivo esperado. Esto mantiene el flujo de trabajo simple y reproducible para los ejercicios de clase.

### Lógica de selección
La carpeta sin formato puede contener varios archivos. Seleccionamos el archivo más nuevo por la marca de tiempo del nombre del archivo para que la canalización siempre use el último scrape.

**Objetivo:** Localizar y cargar el archivo de scrape sin formato más reciente.
**Entradas:** Raíz del repositorio y patrón de nombre de archivo.
**Proceso:** Busque archivos coincidentes, elija el más reciente, lea CSV.
**Salidas:** `df_raw` que contiene datos extraídos sin modificar.
**Por qué:** El raspado más reciente representa el conjunto de datos actual de la clase.


In [ ]:
# Objetivo: localizar y cargar el archivo CSV sin formato más reciente.  #Objetivo
# Entrada: rutas del repositorio y patrón de nombre de archivo creados a partir del alcance configurado.  #Aporte
# Proceso: enumera los archivos coincidentes, selecciona el más nuevo y lee CSV.  #Proceso
# Salida: df_raw que contiene listados extraídos sin modificaciones.  #Producción

raw_dir = RAW_DATA_DIR
pattern = f'autoscout24_listings_{make}_{model}_{country_name}_*.csv'
raw_files = sorted(raw_dir.glob(pattern))
if not raw_files:
    raise FileNotFoundError(f'No raw files found matching {pattern} in {raw_dir}')

raw_path = raw_files[-1]
print('Using raw file:', raw_path.name)

df_raw = pd.read_csv(raw_path)


## 3. Verificaciones de esquemas básicos y copia de trabajo

Las comprobaciones de esquema protegen a los estudiantes de errores silenciosos si cambia el formato sin formato CSV.

### Por qué es importante la verificación del esquema
Si el raspado sin procesar cambia sus columnas, los escalones posteriores pueden romperse silenciosamente. Verificamos explícitamente que las columnas coincidan con uno de los dos esquemas esperados antes de continuar.

### Seguridad del esquema
Un esquema fijo protege el resto del cuaderno. Si faltan columnas o son inesperadas, nos detenemos temprano para que los errores sean visibles y fáciles de corregir.

**Objetivo:** Asegúrese de que el esquema de entrada coincida con las columnas esperadas.
**Entradas:** `df_raw` nombres de columnas.
**Proceso:** Comparar con los esquemas permitidos y detener si no coinciden.
**Salidas:** Una copia de trabajo validada `df`.
**Por qué:** Las comprobaciones de esquemas evitan errores silenciosos más adelante en el proceso.


In [ ]:
# Objetivo: crear una copia de trabajo y verificar el esquema de entrada.  #Objetivo
# Entrada: df_raw cargado desde el archivo CSV sin formato.  #Aporte
# Proceso: copie los datos, asegúrese de que exista la marca, verifique la lista de columnas con los esquemas esperados.  #Proceso
# Salida: df (copia de trabajo) y un esquema validado para pasos posteriores seguros.  #Producción

# Trabaje en una copia para preservar el archivo sin formato # Esto mantiene df_raw sin cambios.
df = df_raw.copy()  # Cree un marco de datos funcional.

# Asegúrese de que existe la marca para los pasos posteriores (scrape de marca única) # Agrega compatibilidad.
if 'brand' not in df.columns:  # Si falta la marca,
    df['brand'] = df['make']  # Crear marca a partir de marca.

# Verifique que el archivo sin formato tenga el formato esperado (esquema fijo) # Validar columnas.
expected_columns_default = [  # Esquema esperado para raspados genéricos.
    'make', 'model', 'mileage', 'price', 'registration', 'fuel', 'country', 'brand', 'page'
]
expected_columns_audi = [  # Esquema esperado para el raspado del Audi A3 con campos adicionales.
    'make', 'model', 'mileage', 'price', 'price_label', 'registration', 'fuel', 'country',
    'power_hp', 'country_name', 'page'
]

if list(df_raw.columns) not in [expected_columns_default, expected_columns_audi]:  # Si el esquema no coincide,
    raise ValueError(  # Genera un error para evitar fallas silenciosas.
        f'Unexpected columns. Expected: {expected_columns_default} or {expected_columns_audi} '
        f'| Found: {list(df_raw.columns)}'
    )
else:  # Si el esquema coincide con las definiciones esperadas,
    print('Column names are as expected.')  # Confirmar validación aprobada.


## 4. Pasos básicos de limpieza

Esta sección contiene la lógica de limpieza central utilizada para todos los análisis y modelados posteriores.

### Principios de limpieza
Mantenemos las transformaciones sencillas: convertimos tipos de datos, estandarizamos texto y asignamos códigos a etiquetas legibles. Cada cambio se almacena en columnas nuevas o columnas con nombres claros.

### Objetivos de limpieza
Convertimos campos numéricos, analizamos fechas, normalizamos texto y asignamos valores codificados a etiquetas legibles. Estos son pasos de limpieza estándar para principiantes.

**Objetivo:** Limpiar campos numéricos, analizar fechas y normalizar texto.
**Entradas:** Columnas sin procesar como precio, kilometraje, registro y combustible.
**Proceso:** Convertir tipos, derivar fechas, estandarizar categorías.
**Salidas:** Columnas limpias y campos nuevos como `fuel_clean`.
**Por qué:** Para el análisis se necesitan tipos y categorías coherentes.


In [ ]:
# Objetivo: Limpiar campos centrales y estandarizar valores categóricos.  #Objetivo
# Entrada: df con tipos sin formato y campos de texto.  #Aporte
# Proceso: eliminar filas faltantes, convertir campos numéricos, analizar fechas, estandarizar texto, asignar códigos.  #Proceso
# Salida: df con tipos numéricos, fechas analizadas y columnas categóricas limpias.  #Producción

# Opcional: eliminar filas con valores faltantes (mantenidas para preservar los resultados actuales) # Eliminar filas incompletas.
df = df.dropna()  # Elimine cualquier fila que contenga valores NaN.

# Convertir columnas numéricas; los valores no válidos se convierten en NaN # Garantizar tipos numéricos.
df['price'] = pd.to_numeric(df['price'], errors='coerce')  # Convertir precio a numérico.
df['mileage'] = pd.to_numeric(df['mileage'], errors='coerce')  # Convierta el kilometraje a numérico.
df['power_hp'] = pd.to_numeric(df['power_hp'], errors='coerce')  # Convertir potencia a numérica.

# Mantenga el texto de registro sin formato # Preservar el valor de cadena original.
df['registration_raw'] = df['registration']  # Almacene el texto de registro sin formato.

# Convierta el registro a fecha y extraiga año/mes # Analizar y dividir fecha.
# Formato esperado: MM-AAAA # Defina el formato de entrada esperado.
df['registration_date'] = pd.to_datetime(df['registration'], format='%m-%Y', errors='coerce')  # Analizar fecha.
df['registration_year'] = df['registration_date'].dt.year  # Extraer año de registro.
df['registration_month'] = df['registration_date'].dt.month  # Extraer mes de inscripción.

# Estandarizar columnas de texto # Normalizar cadenas categóricas.
df['make'] = df['make'].astype('string').str.strip().str.lower()  # Normalizar la marca.
df['model'] = df['model'].astype('string').str.strip().str.lower()  # Normalizar el modelo.
df['brand'] = df['brand'].astype('string').str.strip().str.lower()  # Normalizar la marca.
df['fuel'] = df['fuel'].astype('string').str.strip().str.lower()  # Normalizar el combustible.
df['price_label'] = df['price_label'].astype('string').str.strip().str.lower()  # Normalizar la etiqueta de precio.
df['country'] = df['country'].astype('string').str.strip().str.lower()  # Normalizar el país.

# Asigne códigos de combustible a etiquetas legibles # Traduzca códigos cortos.
fuel_mapping = {  # Definir el mapeo de códigos de combustible.
    'd': 'diesel',
    'b': 'petrol',
    'e': 'electric',
    'l': 'lpg',
    'h': 'hybrid',
}

df['fuel_clean'] = df['fuel'].map(fuel_mapping).fillna(df['fuel'])  # Mapear códigos y mantener incógnitas.

# Asigna códigos de países a nombres legibles # Traduce códigos de países.
country_mapping = {  # Definir la asignación de códigos de países.
    'd': 'germany',
    'b': 'belgium',
    'i': 'italy',
    'f': 'france',
    'nl': 'netherlands',
    'e': 'spain',
}

df['country_clean'] = df['country'].map(country_mapping).fillna(df['country'])  # Mapear códigos y mantener incógnitas.


## 5. Duplicados y valores atípicos

El manejo de valores atípicos se mantiene idéntico al original para que los recuentos de filas y los valores coincidan.

### Duplicados y valores atípicos
Eliminamos duplicados exactos y utilizamos la regla IQR para marcar y eliminar valores atípicos. Esto evita que los valores extremos distorsionen análisis posteriores y, al mismo tiempo, se mantiene fiel a las reglas originales.

### Justificación del manejo de valores atípicos
Los valores atípicos pueden influir en gran medida en los promedios y los modelos. Usamos una regla IQR simple para marcarlos y luego eliminarlos para obtener un conjunto de datos más limpio.

**Objetivo:** Eliminar duplicados y filtrar valores atípicos.
**Entradas:** Columnas numéricas para precio, kilometraje y potencia.
**Proceso:** Eliminar duplicados, calcular límites IQR, eliminar filas marcadas.
**Salidas:** Conjunto de datos limpio con indicadores de valores atípicos.
**Por qué:** Los valores atípicos distorsionan los promedios y el entrenamiento de modelos.


In [ ]:
# Objetivo: eliminar duplicados y filtrar valores atípicos utilizando la regla IQR.  #Objetivo
# Entrada: df con campos numéricos para precio, kilometraje y potencia.  #Aporte
# Proceso: eliminar duplicados, calcular límites IQR, crear indicadores de valores atípicos, filtrar filas marcadas.  #Proceso
# Salida: df con banderas atípicas y filas atípicas eliminadas.  #Producción

# Opcional: eliminar duplicados exactos (se mantienen para preservar los resultados actuales) # Eliminar filas idénticas.
df = df.drop_duplicates()  # Elimine filas duplicadas exactas.

# IQR indicadores de valores atípicos # Defina un asistente para detectar valores atípicos.
def iqr_outlier_flags(series):  # Función para marcar valores fuera de 1,5 * IQR.
    q1 = series.quantile(0.25)  # Primer cuartil.
    q3 = series.quantile(0.75)  # Tercer cuartil.
    iqr = q3 - q1  # Rango intercuartil.
    lower = q1 - 1.5 * iqr  # Límite inferior del valor atípico.
    upper = q3 + 1.5 * iqr  # Límite superior del valor atípico.
    return (series < lower) | (series > upper)  # Cierto para los valores atípicos.

# Cree indicadores de precio, kilometraje y potencia. # Agregue columnas booleanas de valores atípicos.
df['price_outlier_iqr'] = iqr_outlier_flags(df['price'])  # Marcar valores atípicos de precios.
df['mileage_outlier_iqr'] = iqr_outlier_flags(df['mileage'])  # Marcar valores atípicos de kilometraje.
df['power_outlier_iqr'] = iqr_outlier_flags(df['power_hp'])  # Marcar valores atípicos de potencia.

# Opcional: eliminar filas marcadas como valores atípicos (mantenidas para preservar los resultados actuales) # Eliminar filas marcadas.
df = df[~(df['price_outlier_iqr'] | df['mileage_outlier_iqr'] | df['power_outlier_iqr'])]  # Mantenga los valores no atípicos.


## 6. Comprobaciones de coherencia lógica

Las reglas lógicas marcan valores imposibles (como fechas de registro futuras) y los eliminan si están presentes.

### Comprobaciones lógicas
Marcamos valores imposibles (precios negativos, registros futuros, etc.) y eliminamos esas filas. Estas reglas son conservadoras y están destinadas a enseñar controles básicos de calidad de datos.

### Justificación de las comprobaciones lógicas
Algunos valores son imposibles (por ejemplo, precio negativo). Marcamos estos problemas y los eliminamos para evitar enseñar con datos claramente incorrectos.

**Objetivo:** Eliminar registros lógicamente inconsistentes.
**Entradas:** Precio, kilometraje, potencia y fechas de registro.
**Proceso:** Marcar valores no válidos y filtrarlos.
**Salidas:** Conjunto de datos sin valores imposibles.
**Por qué:** Las comprobaciones lógicas mejoran la credibilidad de los datos para la enseñanza.


In [ ]:
# Objetivo: Marcar y eliminar registros lógicamente imposibles.  #Objetivo
# Entrada: df con campos numéricos y fechas de registro.  #Aporte
# Proceso: crear indicadores de validez, combinarlos en logic_issue, filtrar filas incorrectas.  #Proceso
# Salida: df con filas lógicamente inconsistentes eliminadas.  #Producción

# Utilice la fecha de hoy para comprobar fechas futuras imposibles # Capture la fecha de referencia.
# (la fecha se captura una vez y se reutiliza para los cálculos de edad) # Mantenga la coherencia.
today = pd.Timestamp.today()  # Almacene la marca de tiempo de hoy.

# Comprobaciones básicas de cordura # Identificar valores no válidos.
df['invalid_price'] = df['price'] <= 0  # Señalar precios no positivos.
df['invalid_mileage'] = df['mileage'] <= 0  # Marcar kilometraje no positivo.
df['invalid_power_hp'] = df['power_hp'] <= 0  # Bandera de poder no positivo.

# Las fechas de registro no deben estar en el futuro # Marcar fechas futuras.
df['future_registration'] = df['registration_date'] > today  # Verdadero si la fecha es futura.

recent_year = today.year - 1  # Defina "reciente" como dentro del último año.
df['recent_with_high_mileage'] = (df['registration_year'] >= recent_year) & (df['mileage'] > 300000)  # Caso sospechoso.

logical_issue_cols = [  # Lista de todas las banderas de problemas lógicos.
    'invalid_price',
    'invalid_mileage',
    'invalid_power_hp',
    'future_registration',
    'recent_with_high_mileage',
]

df['logical_issue'] = df[logical_issue_cols].any(axis=1)  # Verdadero si hay algún problema presente.

# Opcional: eliminar filas con inconsistencias lógicas (mantenidas para preservar los resultados actuales) # Eliminar filas incorrectas.
df = df[~df['logical_issue']]  # Mantenga solo filas sin problemas lógicos.


## 7. Conjunto de datos final y guardar

El conjunto de datos final tiene un esquema fijo utilizado por los cuadernos SQL y ML.

### Conjunto de datos final
Seleccionamos una lista fija de columnas, le cambiamos el nombre para mayor claridad, agregamos una función de edad simple y guardamos el CSV procesado. El esquema de salida es estable para los portátiles posteriores.

### Salida final
Elegimos un conjunto específico de columnas, les cambiamos el nombre de manera consistente, agregamos una característica de edad básica y guardamos el conjunto de datos limpio en el disco.

**Objetivo:** Construir el esquema final y guardar el conjunto de datos procesado.
**Entradas:** Marco de datos limpio con todos los campos derivados.
**Proceso:** Seleccione columnas, cambie el nombre, agregue función de edad, guarde CSV.
**Salidas:** CSV procesado con marca de tiempo en `data/processed`.
**Por qué:** Un esquema estable admite los pasos SQL y ML posteriores.


In [ ]:
# Objetivo: crear el esquema del conjunto de datos final y guardar el CSV procesado.  #Objetivo
# Entrada: df después de la limpieza, eliminación de valores atípicos y comprobaciones lógicas.  #Aporte
# Proceso: seleccione columnas, cambie el nombre, agregue función de edad, emita tipos de claves y escriba resultados.  #Proceso
# Salida: df_final, CSV procesado y un artefacto de resumen de preprocesamiento compacto.  #Producción

columns_to_keep = [
    'make',
    'model',
    'brand',
    'price',
    'price_label',
    'mileage',
    'power_hp',
    'registration_date',
    'registration_year',
    'registration_month',
    'fuel_clean',
    'country_clean',
    'page',
    'price_outlier_iqr',
    'mileage_outlier_iqr',
    'power_outlier_iqr',
    'logical_issue',
]

df_final = df[columns_to_keep].copy()

rename_map = {
    'price': 'price_eur',
    'mileage': 'mileage_km',
    'fuel_clean': 'fuel_type',
    'country_clean': 'listing_country',
}
df_final = df_final.rename(columns=rename_map)

# Característica: antigüedad del automóvil en años # Característica derivada para análisis.
df_final['age_years'] = today.year - df_final['registration_year']

# Manejo de tipos estables para columnas contractuales.
df_final['price_eur'] = pd.to_numeric(df_final['price_eur'], errors='coerce').astype('Int64')
df_final['mileage_km'] = pd.to_numeric(df_final['mileage_km'], errors='coerce').astype('Int64')
df_final['power_hp'] = pd.to_numeric(df_final['power_hp'], errors='coerce')
df_final['registration_year'] = pd.to_numeric(df_final['registration_year'], errors='coerce').astype('Int64')
df_final['registration_month'] = pd.to_numeric(df_final['registration_month'], errors='coerce').astype('Int64')
df_final['age_years'] = pd.to_numeric(df_final['age_years'], errors='coerce')

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = PROCESSED_DATA_DIR / f'autoscout24_listings_processed_{tag}_{timestamp}.csv'

df_final.to_csv(output_path, index=False)
print('Saved to', output_path)

summary_df = pd.DataFrame([
    {
        'rows': len(df_final),
        'columns': len(df_final.columns),
        'null_price_eur': int(df_final['price_eur'].isna().sum()),
        'null_mileage_km': int(df_final['mileage_km'].isna().sum()),
        'null_power_hp': int(df_final['power_hp'].isna().sum()),
    }
])
summary_path = REPORTS_DIR / f'preprocessing_summary_{tag}_{timestamp}.csv'
summary_df.to_csv(summary_path, index=False)
print('Summary saved to', summary_path)
